In [1]:
!pip install -q transformers==4.57.3
!pip install -q accelerate==1.12.0
!pip install -q bitsandbytes==0.49.1
!pip install -q langchain==1.2.3
!pip install -q langchain-community==0.4.1
!pip install -q langchain-text-splitters
!pip install -q pymupdf
!pip install -q langchain-huggingface
!pip install -q chromadb==1.4.1
!pip install -q huggingface-hub==0.36.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [2]:
import os
import torch
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Perangkat yang digunakan: {device}")

Perangkat yang digunakan: cuda


In [4]:
!wget -O buku_panduan_gen_ai.pdf "https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb"

--2026-05-07 09:37:38--  https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.google.com (drive.google.com)... 173.194.202.101, 173.194.202.138, 173.194.202.102, ...
Connecting to drive.google.com (drive.google.com)|173.194.202.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb [following]
--2026-05-07 09:37:38--  https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.99.132, 2607:f8b0:400e:c0c::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.99.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 655208 (640K) [application/octet-stream]
Saving to: ‘buku_panduan_gen_ai.pdf’

buku_panduan_gen_ai 100%[===================>] 639.85K  --.-KB/s    in 0.004s  

2026-05-07 09:37:40

In [5]:
file_path = "/content/buku_panduan_gen_ai.pdf"
loader = PyMuPDFLoader(file_path)
documents = loader.load()

In [7]:
documents[8]

Document(metadata={'producer': 'iLovePDF', 'creator': 'Adobe InDesign 20.3 (Windows)', 'creationdate': '2025-06-03T13:36:46+07:00', 'source': '/content/buku_panduan_gen_ai.pdf', 'file_path': '/content/buku_panduan_gen_ai.pdf', 'total_pages': 42, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-06-04T05:01:35+00:00', 'trapped': '', 'modDate': 'D:20250604050135Z', 'creationDate': "D:20250603133646+07'00'", 'page': 8}, page_content='8\nBAB II: Pemahaman Teknologi Generative AI\n2.1. Perbedaan AI dan Generative AI\nSebelum memahami cara kerja Generative AI (GenAI), penting bagi mahasiswa untuk \nmemahami perbedaan antara Artificial Intelligence (AI) secara umum dan GenAI se-\ncara khusus.\n1.\t Artificial Intelligence (AI)\n•\t AI adalah teknologi yang dirancang untuk meniru kecerdasan manusia dalam berb-\nagai tugas seperti analisis data, pengenalan pola, dan pengambilan keputusan.\n•\t Contoh AI meliputi asisten virtual seperti Siri dan Goog

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "."]
)

# Proses Splitting
chunks = text_splitter.split_documents(documents)
print(f"Dokumen dipecah menjadi {len(chunks)} chunks.")

Dokumen dipecah menjadi 78 chunks.


In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': device}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [10]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

In [11]:
import shutil

# Mengompres folder chroma_db menjadi chroma_db.zip
shutil.make_archive('chroma_db', 'zip', 'chroma_db')
print("Folder berhasil dikompres!")

Folder berhasil dikompres!


In [13]:
from google.colab import files
files.download('chroma_db.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# Mencari yang mirip tapi tetap memberikan variasi informasi
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10} # Cari 10 yang mirip, lalu pilih 3 yang paling beda satu sama lain
)

In [15]:
# 1. Pastikan menggunakan nama variabel yang benar (retriever_mmr)
query = "Apa itu Generative AI"
retrieved_docs = retriever_mmr.invoke(query)

# 2. Menampilkan hasil dengan informasi tambahan (Metadata)
print(f"Hasil pencarian untuk: '{query}'\n")

for i, doc in enumerate(retrieved_docs, start=1):
    # Mengambil nomor halaman jika ada di metadata (biasanya dari PyPDFLoader)
    page_num = doc.metadata.get('page', 'N/A')

    print(f"--- Dokumen {i} (Halaman: {page_num}) ---")
    print(doc.page_content.strip()) # .strip() untuk membersihkan spasi berlebih
    print("-" * 30) # Garis pembatas antar dokumen agar enak dibaca
    print()

Hasil pencarian untuk: 'Apa itu Generative AI'

--- Dokumen 1 (Halaman: 8) ---
8
BAB II: Pemahaman Teknologi Generative AI
2.1. Perbedaan AI dan Generative AI
Sebelum memahami cara kerja Generative AI (GenAI), penting bagi mahasiswa untuk 
memahami perbedaan antara Artificial Intelligence (AI) secara umum dan GenAI se-
cara khusus.
1.	 Artificial Intelligence (AI)
•	 AI adalah teknologi yang dirancang untuk meniru kecerdasan manusia dalam berb-
agai tugas seperti analisis data, pengenalan pola, dan pengambilan keputusan.
•	 Contoh AI meliputi asisten virtual seperti Siri dan Google Assistant, algoritma pen-
carian Google, dan sistem rekomendasi Netflix atau Spotify.
•	 AI umumnya bersifat berbasis aturan atau pembelajaran mesin yang membantu ma-
nusia dalam mengambil keputusan.
2. Generative AI (GenAI)
•	 GenAI adalah cabang dari AI yang mampu menghasilkan konten baru, seperti teks, 
gambar, video, dan suara, berdasarkan data yang telah dipelajarinya.
•	 Model GenAI menggunakan teknik 

In [16]:
# Konfigurasi Kuantisasi 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [17]:
model_name = "unsloth/Llama-3.2-3B-Instruct"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [18]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=1000,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Device set to use cuda:0


In [19]:
# Llama 3 menggunakan format <|start_header_id|>system<|end_header_id|> dst.
template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Anda adalah asisten AI yang bertugas membantu pengguna.
Gunakan hanya informasi yang tersedia pada konteks berikut untuk menjawab pertanyaan.
Jika jawaban tidak ditemukan dalam konteks tersebut, sampaikan dengan jujur bahwa Anda tidak mengetahui jawabannya dan jangan membuat asumsi atau jawaban tambahan.
Berikan jawaban secara singkat dan jelas.

Context:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>

{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "query"]
)

In [21]:
rag_chain = (
    {"context": retriever_mmr, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [24]:
def ask_question(query):
    print(f"Pertanyaan: {query}")

    # Jalankan Chain
    response = rag_chain.invoke(query)

    print("Jawaban:")
    print(response)

    # Tampilkan sumber referensi
    docs = retriever_mmr.invoke(query)
    print("\nSumber Referensi:")
    for i, doc in enumerate(docs):
        print(f"{i+1}. Halaman {doc.metadata.get('page', '?')}")

In [25]:
query = "Apa Manfaat Generative AI bagi Mahasiswa?"
ask_question(query)

Pertanyaan: Apa Manfaat Generative AI bagi Mahasiswa?
Jawaban:
Manfaat Generative AI bagi mahasiswa antara lain:

*   Membantu dalam mencari ide dan referensi saat belajar, menyusun makalah, dan tugas akademik.
*   Mendukung riset dan analisis data dalam proyek akademik.
*   Meningkatkan pemahaman konsep melalui penjelasan otomatis dan interaktif.
*   Mempermudah pembuatan presentasi, infografis, dan materi belajar lainnya.
*   Menyediakan alat bantu dalam pembelajaran bahasa dan penerjemahan.
*   Meringkas referensi, dan membandingkan berbagai referensi, untuk mempermudah memahami topik, dan pokok bahasan.
*   Mengubah ide menjadi kenyataan, menciptakan artikel menarik, merancang gambar artistik, membuat musik atau video kreatif hanya dengan satu perintah.
*   Membuat belajar lebih interaktif, dinamis, dan seru.

Sumber Referensi:
1. Halaman 13
2. Halaman 12
3. Halaman 1
